# QA — Pipeline validation

**QA gates:** weather columns ≥ 80, scored cells ≥ 1,024, physics yield artifact, yield CV warnings.

Automated checks on processed artifacts after notebooks `00`–`14`.

**All logic runs in this notebook** (no external script imports).

Writes `data/processed/pipeline_validation_report.json` — run after notebooks 00–14 before sharing results.

In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path

import polars as pl

ROOT = Path('../..').resolve()
PROC = ROOT / 'data' / 'processed'
OUT = ROOT / 'outputs'
MAPS = ROOT / 'maps'
REPORT_PATH = PROC / 'pipeline_validation_report.json'

checks = []


def check(name, ok, detail=''):
    checks.append({'check': name, 'ok': bool(ok), 'detail': detail})
    status = 'PASS' if ok else 'FAIL'
    print(f'{status}: {name}' + (f' — {detail}' if detail else ''))


# --- Required files ---
required_proc = [
    'weather_cell_features.parquet',
    'weather_quarantine_log.csv',
    'modeling_master_cell.parquet',
    'cells_passing_gateway.parquet',
    'solar_scores_by_cell.parquet',
    'wind_scores_by_cell.parquet',
    'cell_resource_scores.parquet',
    'hybrid_suitability.parquet',
    'cell_p50_yield.parquet',
    'cell_physics_yield.parquet',
    'cell_risk_spread.parquet',
    'cell_composite_score.parquet',
    'physics_validation_report.json',
    'risk_stress_report.json',
]
for f in required_proc:
    check(f'exists:{f}', (PROC / f).exists())

for f in [
    'ny_composite_ranked.geojson',
    'ny_solar_screening.geojson',
    'ny_wind_screening.geojson',
    'ny_hybrid_suitability.geojson',
]:
    check(f'exists:outputs/{f}', (OUT / f).exists())

for f in [
    'developability_ny.html',
    'solar_resource_ny.html',
    'wind_resource_ny.html',
    'composite_score_ny.html',
    'hybrid_seasonal_suitability_ny.html',
    'risk_spread_ny.html',
]:
    check(f'exists:maps/{f}', (MAPS / f).exists())

# --- Row counts ---
solar = pl.read_parquet(PROC / 'solar_scores_by_cell.parquet')
wind = pl.read_parquet(PROC / 'wind_scores_by_cell.parquet')
hybrid = pl.read_parquet(PROC / 'hybrid_suitability.parquet')
composite = pl.read_parquet(PROC / 'cell_composite_score.parquet')
risk = pl.read_parquet(PROC / 'cell_risk_spread.parquet')
p50 = pl.read_parquet(PROC / 'cell_p50_yield.parquet')

n = solar.height
check('solar_wind_same_n', solar.height == wind.height, f'{solar.height} vs {wind.height}')
check('hybrid_same_n', hybrid.height == n, str(hybrid.height))
check('composite_same_n', composite.height == n, str(composite.height))
gateway_n = pl.read_parquet(PROC / 'cells_passing_gateway.parquet').height
check('scored_lte_gateway', n <= gateway_n, f'scored={n} gateway={gateway_n}')
check('scored_cells_gte_baseline', n >= 1024, f'n={n} (v1 baseline 1024)')

# --- notebook 01 weather feature count ---
wx = pl.read_parquet(PROC / 'weather_cell_features.parquet')
wx_cols = len(wx.columns)
check('weather_feature_count_gte_80', wx_cols >= 80, f'cols={wx_cols}')
sw = wx['shortwave_mean'].drop_nulls()
check('daytime_shortwave_min', sw.min() >= 200, f'min={sw.min():.1f} W/m2')
check('daytime_shortwave_max', sw.max() <= 400, f'max={sw.max():.1f} W/m2')

# --- Yield CV (warn when negative) ---
for tech in ('solar', 'wind'):
    p = PROC / f'yield_{tech}_cv_report.json'
    if p.exists():
        rep = json.loads(p.read_text())
        r2 = rep.get('cv_oof', {}).get('r2')
        if r2 is not None and r2 < 0:
            check(f'warn:yield_{tech}_cv_negative', True, f'WARNING r2={r2:.3f} (screening target > 0)')
        else:
            check(f'yield_{tech}_cv_r2', r2 is not None, f'r2={r2}')

# --- Risk spread (Phase 6) ---
risk_rep_path = PROC / 'risk_stress_report.json'
if risk_rep_path.exists():
    spread = json.loads(risk_rep_path.read_text()).get('mean_hybrid_risk_spread')
    check('risk_spread_gt_v1', spread is not None and spread > 0.001, f'mean={spread:.4f}')
    if spread is not None and spread < 0.005:
        check('warn:risk_spread_below_r5_target', True, f'WARNING mean={spread:.4f} (target > 0.005)')

# --- Composite P50 source (notebook 13) ---
comp_rep = PROC / 'composite_score_report.json'
if comp_rep.exists():
    comp = json.loads(comp_rep.read_text())
    check('composite_p50_source_documented', 'roadmap2_p50_source' in comp, comp.get('roadmap2_p50_source', ''))

# --- CV reports (resource) ---
for tech in ('solar', 'wind'):
    p = PROC / f'{tech}_resource_cv_report.json'
    if p.exists():
        rep = json.loads(p.read_text())
        r2 = rep.get('cv_land_adjusted_oof', rep.get('cv_oof', {})).get('r2')
        check(f'{tech}_cv_r2_present', r2 is not None, f'r2={r2}')
        if tech == 'solar':
            check('solar_cv_r2_positive', r2 > 0.5, f'r2={r2:.3f}')
        else:
            check('wind_cv_r2_positive', r2 > 0.3, f'r2={r2:.3f}')

# --- Risk: P90 <= P50 ---
p90_path = PROC / 'cell_p90_yield.parquet'
if p90_path.exists():
    p90 = pl.read_parquet(p90_path)
    for tech in ('solar', 'wind'):
        p50c, p90c = f'{tech}_cf_p50', f'{tech}_cf_p90'
        if p50c in p90.columns and p90c in p90.columns:
            bad = p90.filter(pl.col(p90c) > pl.col(p50c) + 1e-9).height
            check(f'{tech}_p90_lte_p50', bad == 0, f'{bad} cells violate')

# --- Composite score range ---
cs = composite['composite_score']
check('composite_score_range', cs.min() >= 0 and cs.max() <= 100, f'{cs.min():.1f}–{cs.max():.1f}')

# --- Jackpot count ---
if 'jackpot_zone' in hybrid.columns:
    nj = int(hybrid['jackpot_zone'].sum())
    check('jackpot_zones', nj >= 100, f'n={nj}')

# --- Optional population (unused) ---
pop_csv = PROC / 'county_population_density.csv'
geoid_pq = PROC / 'cell_county_geoid.parquet'
check('optional:county_population', pop_csv.exists(), 'run 04b if missing')
check('optional:cell_geoid', geoid_pq.exists(), 'run 04b if missing')
master = pl.read_parquet(PROC / 'modeling_master_cell.parquet')
check(
    'optional:population_in_master',
    'population_density' not in master.columns,
    'population filter off by default (OK if False)',
)

passed = sum(1 for c in checks if c['ok'])
failed = [c for c in checks if not c['ok']]
required_failed = [f for f in failed if not f['check'].startswith('optional:')]
report = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'pipeline_version': 'roadmap2_v2',
    'n_checks': len(checks),
    'n_passed': passed,
    'n_failed': len(failed),
    'all_passed': len(required_failed) == 0,
    'checks': checks,
}
REPORT_PATH.write_text(json.dumps(report, indent=2))
print('\nWrote', REPORT_PATH)
print(f'Summary: {passed}/{len(checks)} passed (required failures: {len(required_failed)})')
if failed:
    print('Failures:', [f['check'] for f in failed])

PASS: exists:weather_cell_features.parquet
PASS: exists:weather_quarantine_log.csv
PASS: exists:modeling_master_cell.parquet
PASS: exists:cells_passing_gateway.parquet
PASS: exists:solar_scores_by_cell.parquet
PASS: exists:wind_scores_by_cell.parquet
PASS: exists:cell_resource_scores.parquet
PASS: exists:hybrid_suitability.parquet
PASS: exists:cell_p50_yield.parquet
PASS: exists:cell_physics_yield.parquet
PASS: exists:cell_risk_spread.parquet
PASS: exists:cell_composite_score.parquet
PASS: exists:physics_validation_report.json
PASS: exists:risk_stress_report.json
PASS: exists:outputs/ny_composite_ranked.geojson
PASS: exists:outputs/ny_solar_screening.geojson
PASS: exists:outputs/ny_wind_screening.geojson
PASS: exists:outputs/ny_hybrid_suitability.geojson
PASS: exists:maps/developability_ny.html
PASS: exists:maps/solar_resource_ny.html
PASS: exists:maps/wind_resource_ny.html
PASS: exists:maps/composite_score_ny.html
PASS: exists:maps/hybrid_seasonal_suitability_ny.html
PASS: exists:maps

PASS: optional:population_in_master — thesis default off (OK if False)

Wrote /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/pipeline_validation_report.json
Summary: 47/47 passed (required failures: 0)
